# 12 - Overlay: ticker mention *first derivative* vs price (AUTO)

Auto-picks the most-mentioned tickers in the window and, for each, plots the
first derivative of attention (change in the 7-day rolling mention total)
against price. No typing needed.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
P = os.path.join(ROOT, 'data', 'processed')
PRICES_PATH = os.path.join(ROOT, 'data', 'prices', 'prices.parquet')

# window comes from update_data.py - the SAME toggle as live vs backtest
from update_data import START_DATE, END_DATE
WIN_LO = pd.to_datetime(START_DATE)
WIN_HI = pd.to_datetime(END_DATE) if END_DATE else None
print('window:', WIN_LO.date(), '->', (WIN_HI.date() if WIN_HI is not None else 'LIVE (newest)'))

def clip_series(s):
    s = s[s.index >= WIN_LO]
    return s if WIN_HI is None else s[s.index <= WIN_HI]

def clip_dates(df, col):
    df = df[df[col] >= WIN_LO]
    return df if WIN_HI is None else df[df[col] <= WIN_HI]

def load_prices():
    if not os.path.exists(PRICES_PATH):
        raise FileNotFoundError('prices.parquet not found - run  python pull_bloomberg_prices.py  first.')
    px = pd.read_parquet(PRICES_PATH); px['date'] = pd.to_datetime(px['date'])
    return px

def price_series(prices, symbol):
    one = prices[prices['symbol'] == symbol].sort_values('date')
    return clip_series(one.set_index('date')['px_last'])


In [ ]:
HOW_MANY = 6     # how many top tickers to plot
WINDOW = 7       # rolling window before the derivative

In [ ]:
counts = pd.read_parquet(os.path.join(P, 'daily_ticker_counts.parquet'))
counts['date'] = pd.to_datetime(counts['date'])
counts = clip_dates(counts, 'date')

totals = counts.groupby('ticker')['mention_count'].sum().sort_values(ascending=False)
tickers = totals.head(HOW_MANY).index.tolist()
print('auto-selected tickers:', tickers)

prices = load_prices()
for ticker in tickers:
    m = (counts[counts['ticker'] == ticker].sort_values('date')
         .set_index('date')['mention_count']).asfreq('D').fillna(0)
    derivative = m.rolling(WINDOW).sum().diff()
    px = price_series(prices, ticker)
    if px.empty:
        print('skip', ticker, '- no price rows'); continue
    fig, ax1 = plt.subplots(figsize=(13, 5))
    ax1.axhline(0, color='black', linewidth=0.6)
    ax1.plot(derivative.index, derivative.values, color='tab:green', linewidth=1.5, label='1st derivative')
    ax1.set_ylabel('change in mentions/day', color='tab:green'); ax1.tick_params(axis='y', labelcolor='tab:green')
    ax2 = ax1.twinx()
    ax2.plot(px.index, px.values, color='tab:red', linewidth=1.6, label='price')
    ax2.set_ylabel('price (USD)', color='tab:red'); ax2.tick_params(axis='y', labelcolor='tab:red')
    ax1.set_title(f'{ticker}: attention first derivative vs price'); ax1.grid(True, alpha=0.3)
    fig.tight_layout(); plt.show()